#MUTLI-LAYER TF-BERT for regression

In [1]:
import torch
import torch.nn as nn
from transformers import BertModel, BertTokenizer
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns
import os
import random

np.random.seed(42)
random.seed(42)
torch.manual_seed(42)



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


os.makedirs('plots', exist_ok=True)

In [ ]:
# Data Loading and Preprocessing
visual_features = np.load("visual_features.npy")
audio_features = np.load("audio_features.npy")
labels_df = pd.read_csv("labels.csv")

def split_data(mode):
    idx = labels_df[labels_df["mode"] == mode].index
    return {
        "text": labels_df.loc[idx, "text"].values,
        "visual": visual_features[idx],
        "audio": audio_features[idx],
        "label": labels_df.loc[idx, "label"].values  # Numerical label for regression
    }

train_data = split_data("train")
test_data = split_data("test")
valid_data = split_data("valid")



In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Dataset and Dataloader
class MultimodalDataset(Dataset):
    def __init__(self, data):
        self.text = data["text"]
        self.visual = torch.FloatTensor(data["visual"])
        self.audio = torch.FloatTensor(data["audio"])
        self.label = torch.FloatTensor(data["label"])  # FloatTensor for regression

    def __len__(self):
        return len(self.text)

    def __getitem__(self, idx):
        text_encoded = tokenizer(
            self.text[idx],
            padding="max_length",
            truncation=True,
            max_length=50,
            return_tensors="pt"
        )
        return {
            "input_ids": text_encoded["input_ids"].squeeze(0),
            "attention_mask": text_encoded["attention_mask"].squeeze(0),
            "visual": self.visual[idx],
            "audio": self.audio[idx],
            "label": self.label[idx]
        }

batch_size = 32
train_loader = DataLoader(MultimodalDataset(train_data), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(MultimodalDataset(test_data), batch_size=batch_size)
valid_loader = DataLoader(MultimodalDataset(valid_data), batch_size=batch_size)



In [ ]:
#  Model Architecture
class TCT(nn.Module):
    def __init__(self, text_dim=768, audio_dim=74, visual_dim=128, num_heads=2):
        super().__init__()
        self.audio_proj = nn.Linear(audio_dim, text_dim)
        self.visual_proj = nn.Linear(visual_dim, text_dim)
        self.multihead_attn = nn.MultiheadAttention(text_dim, num_heads)
        self.layer_norm1 = nn.LayerNorm(text_dim)
        self.layer_norm2 = nn.LayerNorm(text_dim)
        self.ffn = nn.Sequential(
            nn.Linear(text_dim, 4 * text_dim),
            nn.ReLU(),
            nn.Linear(4 * text_dim, text_dim)
        )

    def forward(self, text, audio, visual):
        audio_proj = self.audio_proj(audio).unsqueeze(1)
        visual_proj = self.visual_proj(visual).unsqueeze(1)

        Q = text.transpose(0, 1)
        K = audio_proj.transpose(0, 1)
        V = visual_proj.transpose(0, 1)

        attn_output, _ = self.multihead_attn(Q, K, V)
        attn_output = attn_output.transpose(0, 1)

        output = self.layer_norm1(text + attn_output)
        ffn_output = self.ffn(output)
        output = self.layer_norm2(output + ffn_output)
        return output



In [ ]:
class TF_BERT_Regressor(nn.Module):
    def __init__(self, tcf_start_layer=6):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.tct = TCT()
        self.regressor = nn.Linear(768, 1)  # Single output for regression
        self.tcf_start_layer = tcf_start_layer

    def forward(self, input_ids, attention_mask, visual, audio):
        attention_mask = attention_mask.float().unsqueeze(1).unsqueeze(2)
        text_output = self.bert.embeddings(input_ids)

        for i, layer in enumerate(self.bert.encoder.layer):
            layer_output = layer(text_output, attention_mask=attention_mask)[0]
            text_output = layer_output

            if i >= self.tcf_start_layer:
                text_output = self.tct(text_output, audio, visual)

        pooled_output = text_output.mean(dim=1)
        prediction = self.regressor(pooled_output)
        return prediction.squeeze(-1)  # Remove last dimension to get (batch_size,)



In [ ]:
# Training Setup
model = TF_BERT_Regressor(tcf_start_layer=6).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
criterion = nn.L1Loss()

# Early stopping parameters
patience = 5
best_mae = float('inf')
early_stop_counter = 0



In [ ]:
# Training and Evaluation Functions
def train_epoch(model, dataloader):
    model.train()
    total_loss = 0
    progress_bar = tqdm(dataloader, desc="Training")
    for batch in progress_bar:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        visual = batch["visual"].to(device)
        audio = batch["audio"].to(device)
        labels = batch["label"].to(device)

        predictions = model(input_ids, attention_mask, visual, audio)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        progress_bar.set_postfix({"loss": loss.item()})
    return total_loss / len(dataloader)



In [ ]:
def validate(model, dataloader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validation"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            visual = batch["visual"].to(device)
            audio = batch["audio"].to(device)
            labels = batch["label"].to(device)

            predictions = model(input_ids, attention_mask, visual, audio)
            loss = criterion(predictions, labels)
            total_loss += loss.item()
    return total_loss / len(dataloader)



In [ ]:
def evaluate(model, dataloader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluation"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            visual = batch["visual"].to(device)
            audio = batch["audio"].to(device)
            batch_labels = batch["label"].to(device)

            batch_preds = model(input_ids, attention_mask, visual, audio)
            preds.extend(batch_preds.cpu().numpy())
            labels.extend(batch_labels.cpu().numpy())

    # Calculate metrics
    mae = np.mean(np.abs(np.array(preds) - np.array(labels)))
    pearson = pearsonr(preds, labels)[0]

    return mae, pearson



In [ ]:
# 6. Training Loop
max_epochs = 50
train_losses = []
val_losses = []
val_metrics = {'mae': [], 'pearson': []}

for epoch in range(max_epochs):
    # Training
    epoch_train_loss = train_epoch(model, train_loader)
    train_losses.append(epoch_train_loss)

    # Validation
    epoch_val_loss = validate(model, valid_loader)
    val_losses.append(epoch_val_loss)

    # Evaluation metrics
    mae, pearson = evaluate(model, valid_loader)
    val_metrics['mae'].append(mae)
    val_metrics['pearson'].append(pearson)

    print(f"Epoch {epoch+1}/{max_epochs}: Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | "
          f"Val MAE: {mae:.3f} | Val Pearson: {pearson:.3f}")

    # Early stopping check based on MAE
    if mae < best_mae:
        best_mae = mae
        early_stop_counter = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"New best MAE: {best_mae:.4f} - Saving model")
    else:
        early_stop_counter += 1
        print(f"MAE did not improve. Early stopping counter: {early_stop_counter}/{patience}")
        if early_stop_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs!")
            break

# Load best model
model.load_state_dict(torch.load('best_model.pt'))
print(f"Loaded best model with MAE: {best_mae:.4f}")



In [ ]:
# Plot and save training curves
plt.figure(figsize=(15, 5))

# Loss curves
plt.subplot(1, 2, 1)
epochs = range(1, len(train_losses) + 1)  # Create integer epoch numbers starting from 1
plt.plot(epochs, train_losses, label='Training Loss', color='blue')
plt.plot(epochs, val_losses, label='Validation Loss', color='orange')
plt.xlabel('Epoch')
plt.ylabel('MAE Loss')
plt.title('Training and Validation Loss')
plt.xticks(epochs)  # Set x-axis ticks to be integer epoch numbers
plt.legend()

# Metrics
plt.subplot(1, 2, 2)
plt.plot(epochs, val_metrics['mae'], label='Validation MAE', color='red')
plt.plot(epochs, val_metrics['pearson'], label='Validation Pearson', color='green')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.title('Validation Metrics')
plt.xticks(epochs)  # Set x-axis ticks to be integer epoch numbers
plt.legend()

plt.tight_layout()
plt.savefig('plots/training_metrics.png', dpi=300, bbox_inches='tight')
plt.close()

# 8. Test Evaluation
test_mae, test_pearson = evaluate(model, test_loader)
print(f"\nTest Results:")
print(f"MAE: {test_mae:.3f}")
print(f"Pearson Correlation: {test_pearson:.3f}")



In [ ]:
# Plot predictions vs true values
model.eval()
test_preds, test_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        visual = batch["visual"].to(device)
        audio = batch["audio"].to(device)
        labels = batch["label"].to(device)

        preds = model(input_ids, attention_mask, visual, audio)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

plt.figure(figsize=(8, 6))
plt.scatter(test_labels, test_preds, alpha=0.5)
plt.plot([-3, 3], [-3, 3], 'r--')  # Perfect prediction line
plt.xlabel('True Values')
plt.ylabel('Predictions')
plt.title('True vs Predicted Values')
plt.grid(True)
plt.savefig('plots/true_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.close()

# Save all metrics to CSV
results_df = pd.DataFrame({
    'Metric': ['MAE', 'Pearson Correlation'],
    'Value': [test_mae, test_pearson]
})
results_df.to_csv('plots/test_results.csv', index=False)
print("Test results saved to 'plots/test_results.csv'")